# Demo 04 : Node failures

Run the cells in order and watch. There is nothing to write.

## a · retry-policy

We inject a transient failure of the asset store while a node fetches the sprites of a build. `max_attempts` includes the initial attempt, and the retry filter selects the exceptions that trigger a retry.

In [1]:
from typing_extensions import TypedDict
from langgraph.graph import END, START, StateGraph
from langgraph.runtime import Runtime
from langgraph.types import RetryPolicy


class BuildState(TypedDict):
    game_id: str
    status: str


def fetch_sprites(state: BuildState, runtime: Runtime):
    attempt = runtime.execution_info.node_attempt
    print("attempt:", attempt)
    if attempt < 3:
        raise ConnectionError("asset store temporarily unavailable")
    return {"status": "sprites fetched"}


builder = StateGraph(BuildState)
builder.add_node("fetch_sprites", fetch_sprites, retry_policy=RetryPolicy(
    max_attempts=3, initial_interval=0.01, jitter=False, retry_on=ConnectionError))
builder.add_edge(START, "fetch_sprites")
builder.add_edge("fetch_sprites", END)
print(builder.compile().invoke({"game_id": "CS-01", "status": "pending"}))

attempt: 1
attempt: 2
attempt: 3
{'game_id': 'CS-01', 'status': 'sprites fetched'}


## b · node-timeout

We run async nodes with run and idle limits. Heartbeats refresh the idle clock, and the run limit stays fixed. We catch and print the expected timeout exceptions.

In [2]:
import asyncio
from langgraph.errors import NodeTimeoutError
from langgraph.types import TimeoutPolicy


async def slow_fetch(state: BuildState, runtime: Runtime):
    try:
        await asyncio.sleep(0.5)
        return {"status": "late"}
    finally:
        print("slow node: cancelled or finished")


async def progressing_fetch(state: BuildState, runtime: Runtime):
    for sprite in range(4):
        await asyncio.sleep(0.03)
        runtime.heartbeat()
    return {"status": "completed with heartbeats"}


async def timed(node, policy):
    builder = StateGraph(BuildState)
    builder.add_node("fetch_sprites", node, timeout=policy)
    builder.add_edge(START, "fetch_sprites")
    builder.add_edge("fetch_sprites", END)
    try:
        print(await builder.compile().ainvoke({"game_id": "CS-01", "status": "pending"}))
    except NodeTimeoutError as exc:
        print("timeout:", exc.kind, "node:", exc.node)


await timed(slow_fetch, TimeoutPolicy(run_timeout=0.08))
await timed(slow_fetch, TimeoutPolicy(idle_timeout=0.08, refresh_on="heartbeat"))
await timed(progressing_fetch, TimeoutPolicy(idle_timeout=0.08, refresh_on="heartbeat"))
await timed(progressing_fetch, TimeoutPolicy(run_timeout=0.08, idle_timeout=0.08,
                                             refresh_on="heartbeat"))

try:
    invalid = StateGraph(BuildState)
    invalid.add_node("fetch_sprites", fetch_sprites, timeout=0.08)
    invalid.add_edge(START, "fetch_sprites")
    invalid.compile()
except ValueError as exc:
    print("sync timeout:", str(exc).splitlines()[0])

slow node: cancelled or finished
timeout: run node: fetch_sprites


slow node: cancelled or finished
timeout: idle node: fetch_sprites


{'game_id': 'CS-01', 'status': 'completed with heartbeats'}


timeout: run node: fetch_sprites
sync timeout: Node timeouts are only supported for async nodes because sync Python execution cannot be safely cancelled in-process. Node 'fetch_sprites' is sync.


## c · error-handler

We route a failure that has exhausted its retries to a node that builds with placeholder sprites. The handler receives state and an injected `NodeError`. A failed attempt's state writes are discarded, but external side effects need their own protection.

In [3]:
from langgraph.errors import NodeError
from langgraph.types import Command


def unavailable(state: BuildState, runtime: Runtime):
    print("attempt:", runtime.execution_info.node_attempt)
    raise ConnectionError("asset store unavailable")


def on_error(state: BuildState, error: NodeError) -> Command:
    print("handler:", error.node, type(error.error).__name__)
    return Command(update={"status": "placeholder sprites"}, goto="placeholders")


def placeholders(state: BuildState):
    print("build:", state["game_id"], state["status"])
    return {}


builder = StateGraph(BuildState)
builder.add_node("fetch_sprites", unavailable, error_handler=on_error,
                 retry_policy=RetryPolicy(max_attempts=2, retry_on=ConnectionError,
                                          initial_interval=0.01, jitter=False))
builder.add_node("placeholders", placeholders)
builder.add_edge(START, "fetch_sprites")
builder.add_edge("fetch_sprites", END)
builder.add_edge("placeholders", END)
print(builder.compile().invoke({"game_id": "CS-01", "status": "pending"}))

attempt: 1
attempt: 2
handler: fetch_sprites ConnectionError
build: CS-01 placeholder sprites
{'game_id': 'CS-01', 'status': 'placeholder sprites'}


We combine an async timeout with the default retry filter and the same recovery handler. Each retry gets a fresh timeout budget, and `NodeTimeoutError` reaches the handler after retries are exhausted.

In [4]:
async def timed_fetch(state: BuildState, runtime: Runtime):
    print("timed attempt:", runtime.execution_info.node_attempt)
    await asyncio.sleep(0.5)
    return {"status": "late"}


builder = StateGraph(BuildState)
builder.add_node("fetch_sprites", timed_fetch, timeout=TimeoutPolicy(run_timeout=0.08),
                 retry_policy=RetryPolicy(max_attempts=2, initial_interval=0.01, jitter=False),
                 error_handler=on_error)
builder.add_node("placeholders", placeholders)
builder.add_edge(START, "fetch_sprites")
builder.add_edge("fetch_sprites", END)
builder.add_edge("placeholders", END)
print(await builder.compile().ainvoke({"game_id": "CS-01", "status": "pending"}))

timed attempt: 1


timed attempt: 2


handler: fetch_sprites NodeTimeoutError
build: CS-01 placeholder sprites
{'game_id': 'CS-01', 'status': 'placeholder sprites'}


## Things to try

Optional : we can change the game, the asset or the question and run the example again.